In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import UEG_response as ur
import json

from UEG_response import _Maldague_chi_2_0_CV, _reduced_FD

In [ ]:
# Conditions
rs = 3.23
theta = 1.0

# Units
hbar = 1.0
aB = 1.0
m = 1.0
e = 1.0
eps0 = 1/(4*np.pi)

# Normalisation
qF = (9*np.pi/4)**(1/3) / (rs*aB)
EF = hbar**2 * qF**2 / (2*m)
beta = 1/(theta*EF)
n = 3/(4*np.pi*rs**3)
beta_eff = beta / np.sqrt(1 + (1/theta)**2 )

# Tolerances
reltol = 1e-16
abstol = 1e-8
eta_log  = 1e-6
eta_sqrt = 1e-6
eta_pol = 1e-4
points_n = 5
tol_upper = 1e-8
dx = 1e-4
lower = 1e-6
limit = 50
ms = 2


In [ ]:
# Import NN implementation of static LFC
from nn_LFC import G_nn

def G_linear(omega, k):
  return G_nn(k/qF, rs, theta)

def no_G_linear(omega, k):
  return 0.0

def find_eps_equal_0(omega, k, G_linear):
    tmp = 1 - (e**2/(eps0*k**2)) * (1 - G_linear(0.0,k)) * np.real(ur.ideal_linear_response(omega, k, m, hbar, n, beta, ms=ms, reltol=reltol, abstol=abstol, eta_log=eta_log, tol_upper=tol_upper, points_n=points_n, force_output=True))
    idxs,  = np.where( tmp[0:-1]*tmp[1:] < 0.0)

    return (omega[idxs] + omega[idxs+1])/2

In [ ]:
# Plot 2D plot of the dynamic response
# Plot settings
top_val_real =  0.50
top_val_imag =  0.50


# fig, axs = plt.subplots(3, 3, sharex=True, sharey=True, figsize=(3*4.8, 3*4.8))
scale = 0.73
fig, axs = plt.subplots(2, 3, sharex=True, figsize=(scale*(3*4.8 + 2.5), scale * 2*4.8), layout='constrained')


# Inputs
omega1 = np.linspace(-5.0, 5.0, 249) * EF/hbar
omega2 = np.linspace(-5.0, 5.0, 250) * EF/hbar

OMEGA1, OMEGA2 = np.meshgrid(omega1, omega2)
    
k1 = 0.6*qF
k2 = 0.6*qF
csTheta = 0.0

chi2_0 = ur.ideal_quadratic_response(OMEGA1.flatten(), k1, OMEGA2.flatten(), k2, csTheta, 
                                    m, hbar, n, beta, ms=ms, method='maldague', 
                                    reltol=reltol, abstol=abstol, eta_pol=eta_pol, 
                                    eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                    dx=dx, points_n=points_n, force_output=True)

chi2_RPA = ur.quadratic_response(OMEGA1.flatten(), k1, OMEGA2.flatten(), k2, csTheta, 
                            m, hbar, e, eps0, n, beta, ms=ms, method='maldague', 
                            reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper, 
                            dx=dx, points_n=points_n, force_output=True)

# chi2_G   = ur.quadratic_response(OMEGA1.flatten(), k1, OMEGA2.flatten(), k2, csTheta, 
#                             m, hbar, e, eps0, n, beta, ms=ms, G_linear=G_linear, method='maldague', 
#                             reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper, 
#                             dx=dx, points_n=points_n, force_output=True)


chi2s = [chi2_0, chi2_RPA]#, chi2_G]
model_names = ["Ideal", "RPA"]#, "LFC"]


for j in range(2):
    for i, (chi2, model_name) in enumerate(zip(chi2s, model_names)):
        if (j == 0):
            z = np.real(chi2) / (n*beta_eff**2)
            X = OMEGA1/(EF/hbar)
            Y = OMEGA2/(EF/hbar)
            Z = np.reshape(z, OMEGA1.shape)
            top_val = top_val_real
        else:
            z = np.imag(chi2) / (n*beta_eff**2)
            X = OMEGA1/(EF/hbar)
            Y = OMEGA2/(EF/hbar)
            Z = np.reshape(z, OMEGA1.shape)
            top_val = top_val_imag

        # Plotting
        # pp = axs[j,i].pcolor(X, Y, Z, cmap='plasma', clim=[bot_val, top_val])
        pp = axs[j,i+1].pcolor(X, Y, Z, cmap='seismic', clim=[-top_val, top_val])
        axs[j,i+1].set_aspect('equal')

        if (i == 1):
            omegas = find_eps_equal_0(omega1, k1, no_G_linear)
            if (len(omegas) == 4):
                for omega in omegas[np.array([0, 3])]:
                    axs[j,i].plot(omega*np.ones(shape=omega2.shape)/(EF/hbar), omega2/(EF/hbar), ':k')
            omegas = find_eps_equal_0(omega2, k2, no_G_linear)
            if (len(omegas) == 4):
                for omega in omegas[np.array([0, 3])]:
                    axs[j,i].plot(omega1/(EF/hbar), omega*np.ones(shape=omega1.shape)/(EF/hbar), ':k')
            k12 = np.sqrt(k1**2 + 2*csTheta*k1*k2 + k2**2)
            omegas = find_eps_equal_0(omega1, k12, no_G_linear)
            if (len(omegas) == 4):
                for omega in omegas[np.array([0, 3])]:
                    axs[j,i].plot(omega1/(EF/hbar), (omega - omega1)/(EF/hbar), ':k')

        axs[j,i+1].set_xlim([np.min(X), np.max(X)])
        axs[j,i+1].set_ylim([np.min(Y), np.max(Y)])
        if (j==1):
            axs[j,i+1].set_xlabel(r'$\omega_1$ [$\hbar^{-1} E_F$]', fontsize=15)
        axs[j,i+1].set_ylabel(r'$\omega_2$ [$\hbar^{-1} E_F$]', fontsize=15)
        if (j == 0):
            axs[j,i+1].set_title(r"%s"%(model_name), fontsize=15)

        print(f"j = %d: max value: %g, min value %g"%(j,np.max(z), np.min(z)))

    # axs[j,i+1].set_aspect('equal')
    cbar = fig.colorbar(pp)
    if (j == 0):
        cbar.set_label(r'Re$\{\chi^{(2)}\}$ [$n\beta_{eff}^2$]', fontsize=15)
    else:
        cbar.set_label(r'Im$\{\chi^{(2)}\}$ [$n\beta_{eff}^2$]', fontsize=15)


# Inputs
omega1 = np.linspace(-5.0, 5.0, 151) * EF/hbar
omega2 = 2 * EF/hbar

k1 = 0.6*qF
k2 = 0.6*qF
csTheta = 0.0

chi2_0 = ur.ideal_quadratic_response(omega1, k1, omega2, k2, csTheta, 
                                    m, hbar, n, beta, ms=ms, method='maldague', 
                                    reltol=reltol, abstol=abstol, eta_pol=eta_pol, 
                                    eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                    dx=dx, points_n=points_n, force_output=True)

chi2_RPA = ur.quadratic_response(omega1, k1, omega2, k2, csTheta, 
                            m, hbar, e, eps0, n, beta, ms=ms, method='maldague', 
                            reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper, 
                            dx=dx, points_n=points_n, force_output=True)

chi2_G   = ur.quadratic_response(omega1, k1, omega2, k2, csTheta, 
                            m, hbar, e, eps0, n, beta, ms=ms, G_linear=G_linear, method='maldague', 
                            reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper, 
                            dx=dx, points_n=points_n, force_output=True)


axs[0, 0].plot(omega1/(EF/hbar), np.real(chi2_0)/(n*beta_eff**2), '-k', linewidth=2.0, label=r"Ideal")
axs[1, 0].plot(omega1/(EF/hbar), np.imag(chi2_0)/(n*beta_eff**2), '-r', linewidth=2.0)

axs[0, 0].plot(omega1/(EF/hbar), np.real(chi2_RPA)/(n*beta_eff**2), ':k', linewidth=2.0, label=r"RPA")
axs[1, 0].plot(omega1/(EF/hbar), np.imag(chi2_RPA)/(n*beta_eff**2), ':r', linewidth=2.0)

axs[0, 0].plot(omega1/(EF/hbar), np.real(chi2_G)/(n*beta_eff**2), '-.k', linewidth=2.0, label=r"LFC")
axs[1, 0].plot(omega1/(EF/hbar), np.imag(chi2_G)/(n*beta_eff**2), '-.r', linewidth=2.0)

axs[0, 0].set_ylabel(r"Re$\{\chi^{(2)}\}$ [$n\beta_{eff}^2$]", fontsize=15)
axs[1, 0].set_ylabel(r"Im$\{\chi^{(2)}\}$ [$n\beta_{eff}^2$]", fontsize=15)
axs[1, 0].set_xlabel(r"$\omega_1$ [$\hbar^{-1}E_F$]", fontsize=15)

# axs[0,0].set_aspect('equal')
# axs[1,0].set_aspect('equal')

axs[0,0].set_box_aspect(1)
axs[1,0].set_box_aspect(1)

axs[0, 0].legend(fontsize=15)
fig.subplots_adjust(hspace=0.0)


axs[0,0].text(-4.9, 0.16, "(a)", fontsize=15,  horizontalalignment='left', verticalalignment='top')
axs[0,1].text(-4.9, 4.9,   "(b)", fontsize=15, horizontalalignment='left', verticalalignment='top')
axs[0,2].text(-4.9, 4.9,   "(c)", fontsize=15, horizontalalignment='left', verticalalignment='top')

axs[1,0].text(-4.9, 0.27, "(d)", fontsize=15,  horizontalalignment='left', verticalalignment='top')
axs[1,1].text(-4.9, 4.9,   "(e)", fontsize=15, horizontalalignment='left', verticalalignment='top')
axs[1,2].text(-4.9, 4.9,   "(f)", fontsize=15, horizontalalignment='left', verticalalignment='top')


# plt.savefig("figures/interacting_models.jpg", dpi=600, bbox_inches="tight")

